In [1]:
import re
import os
import numpy as np
import matplotlib as mpl
import matplotlib.cm

from tqdm import tqdm
from typing import Optional, Union, List, Dict, Tuple
from IPython.display import HTML

import tensorflow as tf
import tensorflow.keras as keras
from tensorflow.keras.datasets import imdb
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.preprocessing import sequence
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import SparseCategoricalCrossentropy

from transformers.optimization_tf import WarmUp
from transformers import TFAutoModelForSequenceClassification, AutoTokenizer, PreTrainedTokenizer

from alibi.explainers import IntegratedGradients

2026-06-10 07:08:37.017183: I tensorflow/core/util/port.cc:111] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-06-10 07:08:37.234273: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2026-06-10 07:08:38.355338: E tensorflow/compiler/xla/stream_executor/cuda/cuda_dnn.cc:9342] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-06-10 07:08:38.355456: E tensorflow/compiler/xla/stream_executor/cuda/cuda_fft.cc:609] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-06-10 07:08:38.362791: E tensorflow/compiler/xla/stream_executor/cuda/cuda_blas.cc:1518] Unable to register cuBLAS factory: Attempting to regi

In [2]:
best_model_path = "/home/pajaro/explicabilidad_transformer/model"

In [3]:
#cargar el tokenizador
tokenizer = AutoTokenizer.from_pretrained(best_model_path)

In [4]:
#cargar el modelo
tf_model = TFAutoModelForSequenceClassification.from_pretrained(best_model_path, from_pt=True)
print(tf_model.summary())

All PyTorch model weights were used when initializing TFRobertaForSequenceClassification.

All the weights of TFRobertaForSequenceClassification were initialized from the PyTorch model.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFRobertaForSequenceClassification for predictions without further training.


Model: "tf_roberta_for_sequence_classification"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 roberta (TFRobertaMainLaye  multiple                  125387520 
 r)                                                              
                                                                 
 classifier (TFRobertaClass  multiple                  592130    
 ificationHead)                                                  
                                                                 
Total params: 125979650 (480.57 MB)
Trainable params: 125979650 (480.57 MB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________
None


In [5]:
class AutoModelWrapper(keras.Model):
    def __init__(self, transformer: keras.Model, **kwargs):
        """
        Constructor.
        
        Parameters
        ----------
        transformer
            Transformer to be wrapped.
        """
        super().__init__()
        self.transformer = transformer

    def call(self, 
             input_ids: Union[np.ndarray, tf.Tensor], 
             attention_mask: Optional[Union[np.ndarray, tf.Tensor]] = None,
             training: bool = False):
        """
        Performs forward pass throguh the model.
        
        Parameters
        ----------
        input_ids
            Indices of input sequence tokens in the vocabulary.
        attention_mask
            Mask to avoid performing attention on padding token indices.
        
        Returns
        -------
        Classification probabilities.
        """
        out = self.transformer(input_ids=input_ids, attention_mask=attention_mask, training=training)
        return tf.nn.softmax(out.logits, axis=-1)
    
    def get_config(self):
        return {}

    @classmethod
    def from_config(cls, config):
        return cls(**config)

In [6]:
auto_model = AutoModelWrapper(tf_model)

In [7]:
auto_model.layers

In [8]:
auto_model.layers[0].layers

In [9]:
#  Extracting the embeddings layer
layer = auto_model.layers[0].layers[0].embeddings

In [10]:
print(layer)

In [11]:
n_steps = 10
internal_batch_size = 1
method = "gausslegendre"

# define Integrated Gradients explainer (memory-safe config)
ig  = IntegratedGradients(auto_model,
                          layer=layer,
                          n_steps=n_steps,
                          method=method,
                          internal_batch_size=internal_batch_size)

Layer not in the list of `model.layers`. Passing the layer directly would not permit the serialization of the explainer. This is due to nested layers. To permit the serialization of the explainer, provide the layer as a callable which returns the layer given the model.


In [12]:
# define some text to be explained
text_samples = [
    'Me despierto varias veces por la noche con sensación de ahogo.',
    'Mi pareja dice que dejo de respirar mientras duermo y ronco muy fuerte.',
    'Durante el día tengo mucha somnolencia y cansancio aunque duermo suficientes horas.',
    "Paciente con ronquidos intensos y somnolencia diurna."
]

# process input to be explained (same outputs: X_test and kwargs)
tokenized_samples = tokenizer(
    text_samples,
    add_special_tokens=True,
    padding='max_length',
    max_length=128,
    truncation=True,
    return_attention_mask=True,
    return_tensors='np',
)

X_test = tokenized_samples['input_ids'].astype(np.int32)
kwargs = {
    k: tf.constant(v)
    for k, v in tokenized_samples.items()
    if k != 'input_ids'
}

X_test.shape, kwargs.keys()

((4, 128), dict_keys(['attention_mask']))

In [13]:
# get prediction probabilities and predicted class
proba = auto_model(X_test, **kwargs).numpy()
predictions = np.argmax(proba, axis=1).astype(np.int32)
confidence = np.max(proba, axis=1)

# build a baseline that keeps only special tokens and pads the rest
mask = np.isin(X_test, tokenizer.all_special_ids).astype(np.int32)
baselines = (X_test * mask + tokenizer.pad_token_id * (1 - mask)).astype(np.int32)

# get integrated gradients explanation sample-by-sample (safer memory usage)
explanations = []
for i in range(X_test.shape[0]):
    xi = X_test[i:i+1]
    ki = {k: v[i:i+1] for k, v in kwargs.items()}
    bi = baselines[i:i+1]
    ti = predictions[i:i+1]
    exp_i = ig.explain(
        xi,
        forward_kwargs=ki,
        baselines=bi,
        target=ti,
    )
    explanations.append(exp_i)

# quick check of outputs
label_map = {0: 'No apnea', 1: 'Apnea'}
results = [
    {
        'text': text_samples[i],
        'predicted_class': int(predictions[i]),
        'predicted_label': label_map.get(int(predictions[i]), f'class_{int(predictions[i])}'),
        'confidence': float(confidence[i]),
    }
    for i in range(len(text_samples))
]

results, [e.attributions[0].shape for e in explanations]

([{'text': 'Me despierto varias veces por la noche con sensación de ahogo.',
   'predicted_class': 0,
   'predicted_label': 'No apnea',
   'confidence': 0.5764448046684265},
  {'text': 'Mi pareja dice que dejo de respirar mientras duermo y ronco muy fuerte.',
   'predicted_class': 0,
   'predicted_label': 'No apnea',
   'confidence': 0.5894066691398621},
  {'text': 'Durante el día tengo mucha somnolencia y cansancio aunque duermo suficientes horas.',
   'predicted_class': 0,
   'predicted_label': 'No apnea',
   'confidence': 0.5582343935966492},
  {'text': 'Paciente con ronquidos intensos y somnolencia diurna.',
   'predicted_class': 0,
   'predicted_label': 'No apnea',
   'confidence': 0.5667861104011536}],
 [(1, 128, 768), (1, 128, 768), (1, 128, 768), (1, 128, 768)])

In [14]:
# Get attributions values from the explanation objects
attrs = np.concatenate([e.attributions[0] for e in explanations], axis=0)
print('Attributions shape:', attrs.shape)

Attributions shape: (4, 128, 768)


In [15]:
attrs = attrs.sum(axis=2)
print('Attributions shape:', attrs.shape)

Attributions shape: (4, 128)


In [17]:
def  hlstr(string: str , color: str = 'white') -> str:
    """
    Return HTML markup highlighting text with the desired color.
    """
    return f"<mark style=background-color:{color}>{string} </mark>"


def colorize(attrs: np.ndarray, cmap: str = 'PiYG') -> List:
    """
    Compute hex colors based on the attributions for a single instance.
    Uses a diverging colorscale by default and normalizes and scales
    the colormap so that colors are consistent with the attributions.
    
    Parameters
    ----------
    attrs
        Attributions to be visualized.
    cmap
        Matplotlib cmap type.
    """
    cmap_bound = np.abs(attrs).max()
    norm = mpl.colors.Normalize(vmin=-cmap_bound, vmax=cmap_bound)
    cmap = mpl.cm.get_cmap(cmap)
    return list(map(lambda x: mpl.colors.rgb2hex(cmap(norm(x))), attrs))


def display(X: np.ndarray, 
            attrs: np.ndarray, 
            tokenizer: PreTrainedTokenizer,
            pred: np.ndarray) -> None:
    """
    Display the attribution of a given instance.
    
    Parameters
    ----------
    X
        Instance to display the attributions for.
    attrs
        Attributions values for the given instance.
    tokenizer
        Tokenizer to be used for decoding.
    pred
        Classification label (prediction) for the given instance.
    """
    pred_dict = {1: 'Positive review', 0: 'Negative review'}
    
    # remove padding
    fst_pad_indices = np.where(X ==tokenizer.pad_token_id)[0]
    if len(fst_pad_indices) > 0:
        X, attrs = X[:fst_pad_indices[0]], attrs[:fst_pad_indices[0]]
    
    # decode tokens and get colors
    tokens = [tokenizer.decode([X[i]]) for i in range(len(X))]
    colors = colorize(attrs)
    
    print(f'Predicted label =  {pred}: {pred_dict[pred]}')
    return HTML("".join(list(map(hlstr, tokens, colors))))

In [18]:
index = 2
display(X=X_test[index], attrs=attrs[index], pred=predictions[index], tokenizer=tokenizer)

Predicted label =  0: Negative review


The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
